# ETL Pipeline: PDDikti → Star Schema Data Warehouse
## Universitas Siliwangi — Analisis Rasio Dosen:Mahasiswa

**Notebook ini mendokumentasikan proses ETL (Extract → Transform → Load) sebagai bagian dari implementasi BAB 4.**

| Fase | Kegiatan |
|------|----------|
| Extract | Membaca data mentah hasil scraping PDDikti |
| Transform | Cleaning, parsing, normalisasi format |
| Load | Menyimpan ke Star Schema (4 tabel) + flat table untuk BI |

---
**Sumber Data:** PDDikti (https://pddikti.kemdiktisaintek.go.id)  
**Periode:** Ganjil 2023 — Ganjil 2025 (5 periode)  
**Scope:** Universitas Siliwangi (semua program studi aktif)

---
## 0. Import Library

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Path root project (satu level di atas folder Notebooks/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"Root project: {ROOT}")

# Definisi path
PATH_RAW_PRODI   = os.path.join(ROOT, 'Data', 'Processed', 'unsil_prodi_fresh.csv')
PATH_RAW_UNIV    = os.path.join(ROOT, 'Data', 'Processed', 'unsil_univ_fresh.csv')
PATH_OUT_SCHEMA  = os.path.join(ROOT, 'Data', 'Star_Schema')
PATH_OUT_MASTER  = os.path.join(ROOT, 'Data', 'Processed', 'master_looker_unsil.csv')

os.makedirs(PATH_OUT_SCHEMA, exist_ok=True)
print("✅ Library dan path siap.")

---
## 1. EXTRACT — Membaca Data Mentah

Data diperoleh dari scraping PDDikti menggunakan Selenium (`Scripts/Scraping/scrape_unsil_only.py`).  
Format data mentah masih berupa teks apa adanya dari website PDDikti — belum ada transformasi apapun.

In [ ]:
# Baca data mentah hasil scraping
df_prodi_raw = pd.read_csv(PATH_RAW_PRODI)
df_univ_raw  = pd.read_csv(PATH_RAW_UNIV)

print("=" * 60)
print("DATA MENTAH (SEBELUM TRANSFORMASI)")
print("=" * 60)
print(f"\nFile prodi  : {PATH_RAW_PRODI}")
print(f"Jumlah baris: {len(df_prodi_raw)} baris")
print(f"Kolom       : {df_prodi_raw.columns.tolist()}")
print(f"\nFile univ   : {PATH_RAW_UNIV}")
print(f"Jumlah baris: {len(df_univ_raw)} baris")

In [ ]:
# Preview data SEBELUM transformasi (tabel dokumentasi untuk BAB 4)
print("\n[TABEL 1] Sample Data Mentah Prodi (5 baris pertama):")
display(df_prodi_raw.head(5))

print("\n[TABEL 2] Data Universitas Siliwangi (metadata):")
display(df_univ_raw)

In [ ]:
# Statistik data mentah
print("[STATISTIK DATA MENTAH]")
print(f"Total baris prodi (semua periode) : {len(df_prodi_raw)}")
print(f"Periode yang tersedia             : {sorted(df_prodi_raw['tahun_pelaporan'].unique())}")
print(f"Jumlah prodi unik                 : {df_prodi_raw['nama_program_studi'].nunique()}")
print(f"\nNilai kosong per kolom:")
print(df_prodi_raw.isnull().sum()[df_prodi_raw.isnull().sum() > 0])

---
## 2. TRANSFORM — Cleaning & Normalisasi

Tahap ini membersihkan dan menstandarkan data mentah:
1. Hapus baris dengan data kritis yang kosong
2. Parsing kolom `tahun_pelaporan` → `semester` + `tahun`
3. Konversi kolom numerik (`jumlah_mahasiswa`, `total_dosen`, dll.)
4. Parsing kolom `rasio_dosen_mahasiswa` → nilai numerik `nilai_rasio`
5. Standarisasi metadata universitas (kode_pt, status, akreditasi)

In [ ]:
df = df_prodi_raw.copy()

# 1. Hapus baris dengan nilai kritis kosong
before = len(df)
df = df.dropna(subset=['kode_prodi', 'tahun_pelaporan', 'rasio_dosen_mahasiswa'])
after = len(df)
print(f"[1] Drop null kritis: {before} → {after} baris (dihapus {before - after} baris)")

# 2. Parsing tahun_pelaporan → semester + tahun
df[['semester', 'tahun']] = df['tahun_pelaporan'].str.split(' ', n=1, expand=True)
print(f"[2] Parsing periode: {df['tahun_pelaporan'].unique()}")

# 3. Konversi kolom numerik
num_cols = ['jumlah_dosen_penghitung_rasio', 'dosen_tetap', 'dosen_tidak_tetap',
            'total_dosen', 'jumlah_mahasiswa']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(f"[3] Konversi numerik: {num_cols}")

# 4. Parsing rasio 'X:Y.Z' → nilai numerik Y.Z
def parse_rasio(s):
    try:
        if pd.isna(s): return np.nan
        parts = str(s).split(':')
        return float(parts[1]) if len(parts) == 2 else np.nan
    except:
        return np.nan

df['nilai_rasio'] = df['rasio_dosen_mahasiswa'].apply(parse_rasio)
print(f"[4] Parse rasio: contoh '{df['rasio_dosen_mahasiswa'].iloc[0]}' → {df['nilai_rasio'].iloc[0]}")

# 5. Standarisasi metadata universitas
df['kode_pt']         = df['kode_pt'].fillna('002008').replace('-', '002008').replace('', '002008')
df['status_pt_univ']  = df['status_pt_univ'].fillna('PTN').replace('-', 'PTN').replace('Jenjang', 'PTN')
df['akreditasi_pt_univ'] = df['akreditasi_pt_univ'].apply(
    lambda x: 'Unggul' if str(x).startswith('1') or str(x).lower() in ['unggul','-','nan'] else x
).fillna('Unggul')

# Pastikan kode_pt = '002008' untuk semua (kode resmi Unsil)
df['kode_pt'] = '002008'
print(f"[5] Standarisasi metadata Unsil selesai.")

print(f"\n✅ Transformasi selesai: {len(df)} baris siap diproses.")

In [ ]:
# Preview data SETELAH transformasi (tabel dokumentasi untuk BAB 4)
print("[TABEL 3] Sample Data Setelah Transformasi (5 baris pertama):")
cols_tampil = ['tahun_pelaporan', 'semester', 'tahun', 'nama_program_studi', 'jenjang',
               'total_dosen', 'jumlah_mahasiswa', 'rasio_dosen_mahasiswa', 'nilai_rasio']
display(df[cols_tampil].head(5))

print("\n[TABEL 4] Ringkasan Data (untuk tabel skripsi):")
summary = pd.DataFrame({
    'Keterangan': ['Total Baris (semua periode)', 'Jumlah Prodi Unik', 'Jumlah Periode', 
                   'Periode Pertama', 'Periode Terakhir', 'Rasio Terendah (1:x)', 'Rasio Tertinggi (1:x)'],
    'Nilai': [
        len(df),
        df['nama_program_studi'].nunique(),
        df['tahun_pelaporan'].nunique(),
        df['tahun_pelaporan'].min(),
        df['tahun_pelaporan'].max(),
        f"1:{df['nilai_rasio'].min():.2f}",
        f"1:{df['nilai_rasio'].max():.2f}"
    ]
})
display(summary)

---
## 3. LOAD — Pembentukan Star Schema

Berikut struktur Star Schema yang diimplementasikan:

```
                  ┌─────────────────┐
                  │   Dim_Waktu     │
                  │ PK: id_waktu    │
                  │    semester     │
                  │    tahun        │
                  └────────┬────────┘
                           │
  ┌──────────────┐    ┌────┴──────────────────────────┐    ┌────────────────┐
  │ Dim_Universitas│──│   Fact_Kapasitas_Pendidikan   │──  │   Dim_Prodi    │
  │ PK: id_univ   │  │   FK: id_universitas           │    │ PK: id_prodi   │
  │  nama_univ    │  │   FK: id_prodi                 │    │  nama_prodi    │
  │  akreditasi   │  │   FK: id_waktu                 │    │  jenjang       │
  └───────────────┘  │   jumlah_dosen                 │    │  akreditasi    │
                     │   jumlah_mahasiswa              │    └────────────────┘
                     │   nilai_rasio                   │
                     └────────────────────────────────┘
```

### 3a. Dimensi Waktu (Dim_Waktu)

In [ ]:
dim_waktu = (
    df[['tahun_pelaporan', 'semester', 'tahun']]
    .drop_duplicates()
    .sort_values('tahun_pelaporan')
    .reset_index(drop=True)
)
dim_waktu.insert(0, 'id_waktu', dim_waktu.index + 1)
dim_waktu['tahun'] = dim_waktu['tahun'].astype(int)

print("[Dim_Waktu]")
display(dim_waktu)

### 3b. Dimensi Universitas (Dim_Universitas)

In [ ]:
# Ambil dari data univ fresh
row_univ = df_univ_raw.iloc[0]

dim_univ = pd.DataFrame([{
    'id_universitas'     : '002008',
    'nama_universitas'   : 'Universitas Siliwangi',
    'kota'              : row_univ.get('kota', 'Kota Tasikmalaya'),
    'provinsi'          : row_univ.get('provinsi', 'Prov. Jawa Barat'),
    'status_pt'         : 'PTN',
    'akreditasi_institusi': 'Unggul'
}])

print("[Dim_Universitas]")
display(dim_univ)

### 3c. Dimensi Program Studi (Dim_Prodi)

In [ ]:
# Gunakan data periode terbaru sebagai referensi atribut prodi
latest_period = df['tahun_pelaporan'].max()
df_latest = df[df['tahun_pelaporan'] == latest_period]

dim_prodi = (
    df_latest[['kode_prodi', 'nama_program_studi', 'jenjang', 'status_prodi', 'akreditasi_prodi']]
    .drop_duplicates(subset=['kode_prodi'])
    .sort_values('nama_program_studi')
    .reset_index(drop=True)
    .rename(columns={'kode_prodi': 'id_prodi'})
)

print(f"[Dim_Prodi] — {len(dim_prodi)} program studi aktif")
display(dim_prodi)

### 3d. Tabel Fakta (Fact_Kapasitas_Pendidikan)

In [ ]:
# Merge df dengan dim_waktu untuk mendapatkan id_waktu
fact = df.merge(dim_waktu[['id_waktu', 'tahun_pelaporan']], on='tahun_pelaporan', how='left')

fact_table = fact[[
    'kode_pt', 'kode_prodi', 'id_waktu',
    'jumlah_dosen_penghitung_rasio', 'dosen_tetap', 'dosen_tidak_tetap', 'total_dosen',
    'jumlah_mahasiswa', 'rasio_dosen_mahasiswa', 'nilai_rasio'
]].rename(columns={
    'kode_pt'   : 'id_universitas',
    'kode_prodi': 'id_prodi'
})

fact_table = fact_table.dropna(subset=['id_universitas', 'id_prodi']).reset_index(drop=True)

print(f"[Fact_Kapasitas_Pendidikan] — {len(fact_table)} baris")
print(f"Cakupan: {fact_table['id_waktu'].nunique()} periode | {fact_table['id_prodi'].nunique()} prodi unik")
display(fact_table.head(10))

---
## 4. Simpan Star Schema ke File CSV

In [ ]:
dim_waktu.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Dim_Waktu.csv'), index=False)
dim_univ.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Dim_Universitas.csv'), index=False)
dim_prodi.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Dim_Prodi.csv'), index=False)
fact_table.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Fact_Kapasitas_Pendidikan.csv'), index=False)

print("✅ Star Schema tersimpan di:", PATH_OUT_SCHEMA)
for f in ['Dim_Waktu.csv', 'Dim_Universitas.csv', 'Dim_Prodi.csv', 'Fact_Kapasitas_Pendidikan.csv']:
    path = os.path.join(PATH_OUT_SCHEMA, f)
    rows = pd.read_csv(path).shape[0]
    print(f"  {f}: {rows} baris")

---
## 5. Buat Flat Table (master_looker_unsil.csv)

Flat table ini menggabungkan semua dimensi dan fakta menjadi satu tabel yang langsung bisa dipakai untuk visualisasi dashboard.

In [ ]:
master = df[[
    'tahun_pelaporan', 'semester', 'tahun',
    'nama_program_studi', 'jenjang', 'status_prodi', 'akreditasi_prodi',
    'nama_universitas',
    'jumlah_mahasiswa', 'jumlah_dosen_penghitung_rasio', 'dosen_tetap',
    'dosen_tidak_tetap', 'total_dosen',
    'rasio_dosen_mahasiswa', 'nilai_rasio'
]].copy()

master['kota']     = 'Kota Tasikmalaya'
master['provinsi'] = 'Prov. Jawa Barat'
master['kode_pt']  = '002008'

master = master.sort_values(['tahun_pelaporan', 'nama_program_studi']).reset_index(drop=True)
master.to_csv(PATH_OUT_MASTER, index=False)

print(f"✅ master_looker_unsil.csv tersimpan: {len(master)} baris")
print(f"   Prodi : {master['nama_program_studi'].nunique()} prodi unik")
print(f"   Periode: {sorted(master['tahun_pelaporan'].unique())}")
display(master.head())

---
## ✅ ETL Selesai

| Output | File | Keterangan |
|--------|------|------------|
| Dim_Waktu | `Data/Star_Schema/Dim_Waktu.csv` | 5 periode |
| Dim_Universitas | `Data/Star_Schema/Dim_Universitas.csv` | 1 PT (Unsil) |
| Dim_Prodi | `Data/Star_Schema/Dim_Prodi.csv` | Semua prodi aktif |
| Fact Table | `Data/Star_Schema/Fact_Kapasitas_Pendidikan.csv` | Seluruh observasi |
| Flat Table | `Data/Processed/master_looker_unsil.csv` | Untuk visualisasi BI |

**Lanjut ke:** `Notebooks/Dashboard_Visualisasi.ipynb`